# Your Title Here

**Name(s)**: (your name(s) here)

**Website Link**: (your website link)

In [276]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import seaborn as sns
import math

import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
pd.options.plotting.backend = 'plotly'

# from dsc80_utils import * # Feel free to uncomment and use this.

## Step 1: Introduction

In [277]:
# TODO

data = pd.read_csv('outage_cleaned.csv')

## Step 2: Data Cleaning and Exploratory Data Analysis

In [278]:
# TODO
data = pd.read_csv('outage_cleaned.csv')
# Set the observation number to be the index of the dataset
data = data.set_index('OBS')
# Convert the OUTAGE.START.DATE and OUTAGE.START.TIME to datetime
data['OUTAGE.START.DATE'] = pd.to_datetime(data['OUTAGE.START.DATE'] + ' ' + data['OUTAGE.START.TIME'])
# Convert the OUTAGE.RESTORATION.DATE and OUTAGE.RESTORATION.TIME to datetime
data['OUTAGE.RESTORATION.DATE'] = pd.to_datetime(data['OUTAGE.RESTORATION.DATE'] + ' ' + data['OUTAGE.RESTORATION.TIME'])
# Drop the OUTAGE.START.TIME and OUTAGE.RESTORATION.TIME columns
data = data.drop(columns=['OUTAGE.START.TIME', 'OUTAGE.RESTORATION.TIME'])
# Drop the US state column as it is redundant
data = data.drop(columns=['U.S._STATE'])
# data['DATE'] = pd.to_datetime(dict(year=data['YEAR'], month=data['MONTH'], day=1))
# data.drop(columns=['YEAR', 'MONTH'], inplace=True)
data

C:\Users\sweek\AppData\Local\Temp\ipykernel_48336\1784923662.py:6: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

C:\Users\sweek\AppData\Local\Temp\ipykernel_48336\1784923662.py:8: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



,YEAR,MONTH,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,OUTAGE.RESTORATION.DATE,CAUSE.CATEGORY,...,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
OBS,,,,,,,,,,,,,,,,,,,,,
1,2011,7.0,MN,MRO,East North Central,-0.3,normal,2011-07-01 17:00:00,2011-07-03 20:00:00,severe weather,...,73.27,15.28,2279.0,1700.5,18.2,2.14,0.60,91.592666,8.407334,5.478743
2,2014,5.0,MN,MRO,East North Central,-0.1,normal,2014-05-11 18:38:00,2014-05-11 18:39:00,intentional attack,...,73.27,15.28,2279.0,1700.5,18.2,2.14,0.60,91.592666,8.407334,5.478743
3,2010,10.0,MN,MRO,East North Central,-1.5,cold,2010-10-26 20:00:00,2010-10-28 22:00:00,severe weather,...,73.27,15.28,2279.0,1700.5,18.2,2.14,0.60,91.592666,8.407334,5.478743
4,2012,6.0,MN,MRO,East North Central,-0.1,normal,2012-06-19 04:30:00,2012-06-20 23:00:00,severe weather,...,73.27,15.28,2279.0,1700.5,18.2,2.14,0.60,91.592666,8.407334,5.478743
5,2015,7.0,MN,MRO,East North Central,1.2,warm,2015-07-18 02:00:00,2015-07-19 07:00:00,severe weather,...,73.27,15.28,2279.0,1700.5,18.2,2.14,0.60,91.592666,8.407334,5.478743
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1530,2011,12.0,ND,MRO,West North Central,-0.9,cold,2011-12-06 08:00:00,2011-12-06 20:00:00,public appeal,...,59.90,19.90,2192.2,1868.2,3.9,0.27,0.10,97.599649,2.401765,2.401765
1531,2006,NaN,ND,MRO,West North Central,NaN,NaN,NaT,NaT,fuel supply emergency,...,59.90,19.90,2192.2,1868.2,3.9,0.27,0.10,97.599649,2.401765,2.401765
1532,2009,8.0,SD,RFC,West North Central,0.5,warm,2009-08-29 22:54:00,2009-08-29 23:53:00,islanding,...,56.65,26.73,2038.3,1905.4,4.7,0.30,0.15,98.307744,1.692256,1.692256


In [279]:
def plot_against_categorical_var(categorical_var_col, column_name, exclude_top_pct=0.05):
    threshold = data[column_name].quantile(1 - exclude_top_pct)  # exclude top X%
    filtered = data[data[column_name] <= threshold]
    
    fig = px.box(filtered, x=categorical_var_col, y=column_name,
                title=f'{column_name} vs {categorical_var_col} (excluding top {exclude_top_pct*100:.0f}% values)')
    fig.show()


def plot_against_numerical_var(numerical_var_col, column_name, exclude_top_pct=0.05):
    threshold = data[column_name].quantile(1 - exclude_top_pct)
    filtered = data[data[column_name] <= threshold]
    
    fig = px.scatter(filtered, x=numerical_var_col, y=column_name, trendline='ols', trendline_color_override='red',
                    title=f'{column_name} vs {numerical_var_col} (excluding top {exclude_top_pct*100:.0f}% values)')
    fig.show()

def plot_bivariate(column, categorical = True, exclude_top_pct=0.05, save_folder="bivariate_analysis", df=data):
    metrics = ['OUTAGE.DURATION', 'DEMAND.LOSS.MW', 'CUSTOMERS.AFFECTED']
    subplot_titles = metrics

    os.makedirs(save_folder, exist_ok=True)  # ensure folder exists
    print(f"Plotting {column}...")
    
    # Create 1 row, 3 columns
    fig = make_subplots(rows=1, cols=3, subplot_titles=subplot_titles)
    
    for i, metric in enumerate(metrics):
        # Filter out top fraction
        threshold = df[metric].quantile(1 - exclude_top_pct)
        filtered = df[df[metric] <= threshold]
        
        if categorical:
            # Boxplot for categorical x-axis
            for category in filtered[column].unique():
                fig.add_trace(
                    go.Box(
                        y=filtered[filtered[column] == category][metric],
                        name=str(category),
                        boxmean='sd',
                        showlegend=(i==0)  # show legend only for first subplot
                    ),
                    row=1, col=i+1
                )
        else:
            # Scatterplot with line of best fit
            scatter_fig = px.scatter(filtered, x=column, y=metric)
            for trace in scatter_fig.data:
                fig.add_trace(trace, row=1, col=i+1)
    
    fig.update_layout(
        height=500, width=1800,
        title_text=f"Metrics vs {column} (excluding top {exclude_top_pct*100:.0f}% values)"
    )
    
    print(f"Finished plotting {column}.")

    return fig

def plot_univariate(column_name, categorical=False, df=data):
    filtered = df[[column_name]].copy()
    
    if categorical:
        # Categorical: histogram/bar chart
        fig = go.Figure()
        fig.add_trace(
            go.Histogram(
                x=filtered[column_name],
                name=f'{column_name} counts',
            )
        )
        fig.update_layout(
            title_text=f'Histogram of {column_name}',
            xaxis_title=column_name,
            yaxis_title='Count',
            width=600, height=400
        )
    else:
        values = filtered[column_name].values
        
        kde_fig = sns.kdeplot(values, bw_method='scott')
        kde_data = kde_fig.get_lines()[0].get_data()
        x_grid = kde_data[0]
        kde_values = kde_data[1]
        kde_fig.figure.clear() 

        fig = go.Figure()
        fig.add_trace(
            go.Scatter(
                x=x_grid,
                y=kde_values,
                fill='tozeroy',
                name='KDE'
            )
        )
        fig.update_layout(
            width=600, height=400,
            title_text=f'KDE of {column_name}',
            xaxis_title=column_name,
            yaxis_title='Density',
            showlegend=False
        )
    
    return fig


def save_figure(fig, column_name, save_folder="univariate_analysis"):
    os.makedirs(save_folder, exist_ok=True)
    save_path = os.path.join(save_folder, f"{column_name}.png")
    fig.write_image(save_path)
    print(f"Saved figure to {save_path}")


In [280]:
def save_combined_univariate(columns_to_plot, save_path="univariate_analysis/combined.png", cols_per_row=5):
    n_cols = cols_per_row
    n_rows = math.ceil(len(columns_to_plot) / n_cols)
    
    # Create subplot titles
    subplot_titles = [col for col, _ in columns_to_plot]
    
    fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=subplot_titles)
    
    for i, (column_name, categorical) in enumerate(columns_to_plot):
        row = i // n_cols + 1
        col = i % n_cols + 1
        
        # Create figure for this column
        col_fig = plot_univariate(column_name, categorical=categorical)
        
        # Add traces to the combined figure
        for trace in col_fig.data:
            fig.add_trace(trace, row=row, col=col)
        
        # Adjust x-axis and y-axis titles for each subplot
        fig.update_xaxes(title_text=column_name, row=row, col=col)
        fig.update_yaxes(title_text='Count' if categorical else 'Density', row=row, col=col)
    
    fig.update_layout(
        height=n_rows * 400,  # 400px per row
        width=n_cols * 600,   # 600px per column
        title_text="Combined Univariate Analysis",
        showlegend=False
    )
    
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    fig.write_image(save_path)
    print(f"Saved combined figure to {save_path}")

In [281]:
columns_to_plot = [
    # Time of year
    ('MONTH', True),

    # Geographic region information
    ('NERC.REGION', True),
    ('POSTAL.CODE', True),

    # Regional Climate Information
    ('CLIMATE.REGION', True),
    ('CLIMATE.CATEGORY', True),
    ('ANOMALY.LEVEL', False),

    # Event causes
    ('CAUSE.CATEGORY', True),
    ('HURRICANE.NAMES', True),

    # Electricity price information
    ('RES.PRICE', False),
    ('COM.PRICE', False),
    ('IND.PRICE', False),

    # Electricity consumption information
    ('RES.SALES', False),
    ('COM.SALES', False),
    ('IND.SALES', False),
    ('RES.PERCEN', False),
    ('COM.PERCEN', False),
    ('IND.PERCEN', False),

    # Customers served information
    ('RES.CUSTOMERS', False),
    ('COM.CUSTOMERS', False),
    ('IND.CUSTOMERS', False),

    # Regional economic output information
    ('PC.REALGSP.STATE', False),
    ('PC.REALGSP.USA', False),
    ('PC.REALGSP.REL', False),
    ('PC.REALGSP.CHANGE', False),
    ('UTIL.REALGSP', False),
    ('TOTAL.REALGSP', False),
    ('UTIL.CONTRI', False),
    ('PI.UTIL.OFUSA', False),

    # Population information
    ('POPULATION', False),
    ('POPPCT_URBAN', False),
    ('POPPCT_UC', False),
    ('POPDEN_URBAN', False),
    ('POPDEN_UC', False),
    ('POPDEN_RURAL', False),

    # Land information
    ('AREAPCT_URBAN', False),
    ('AREAPCT_UC', False),
    ('PCT_LAND', False),
    ('PCT_WATER_TOT', False),
    ('PCT_WATER_INLAND', False)
]

output_columns = [('OUTAGE.DURATION', False), ('DEMAND.LOSS.MW', False), ('CUSTOMERS.AFFECTED', False)]

# for col, is_categorical in columns_to_plot:
#     fig1 = plot_bivariate(col, categorical=is_categorical)
#     fig2 = plot_univariate(col, categorical=is_categorical)
#     save_figure(fig1, col, save_folder="bivariate_analysis")
#     save_figure(fig2, col, save_folder="univariate_analysis")

# for col, is_categorical in output_columns:
#     fig = plot_univariate(col, categorical=is_categorical)
#     save_figure(fig, col, save_folder="univariate_analysis")

# save_combined_univariate(columns_to_plot + output_columns, save_path="univariate_analysis/all_univariate.png", cols_per_row=5)

In [282]:
threshold = data['OUTAGE.DURATION'].quantile(0.90)
filtered = data[data['OUTAGE.DURATION'] <= threshold]
pivot_mean_outage_duration = filtered.pivot_table(index='CAUSE.CATEGORY', columns='CLIMATE.REGION', values='OUTAGE.DURATION', aggfunc='mean')
pivot_mean_outage_duration.fillna(0, inplace=True)

def pivot_and_heatmap(df, index_col, columns_col, values_col, aggfunc='mean', exclude_top_pct=0.10):
    threshold = df[values_col].quantile(1 - exclude_top_pct)
    filtered = df[df[values_col] <= threshold]
    pivot_table = filtered.pivot_table(index=index_col, columns=columns_col, values=values_col, aggfunc=aggfunc)
    pivot_table.fillna(0, inplace=True)
    fig = px.imshow(
        pivot_table,
        labels=dict(x=columns_col, y=index_col, color=values_col),
        x=pivot_table.columns,
        y=pivot_table.index,
        text_auto=True,
        aspect='auto',
        color_continuous_scale='Viridis'
    )
    fig.update_layout(
        title=f'{values_col} {aggfunc} by {index_col} and {columns_col}, excluding top {exclude_top_pct*100:.0f}%',
        xaxis_title=columns_col,
        yaxis_title=index_col,
        height=600,
        width=900
    )
    return fig

save_figure(
    pivot_and_heatmap(data, index_col='CAUSE.CATEGORY', columns_col='CLIMATE.REGION', values_col='OUTAGE.DURATION', aggfunc='mean', exclude_top_pct=0.10),
    'heatmap_mean_outage_duration',
    save_folder='heatmap_analysis'
)
save_figure(
    pivot_and_heatmap(data, index_col='CAUSE.CATEGORY', columns_col='CLIMATE.REGION', values_col='OUTAGE.DURATION', aggfunc='count', exclude_top_pct=0),
    'heatmap_count_outage_duration',
    save_folder='heatmap_analysis'
)

Saved figure to heatmap_analysis\heatmap_mean_outage_duration.png
Saved figure to heatmap_analysis\heatmap_count_outage_duration.png


## Step 3: Assessment of Missingness

In [283]:
missing_counts = data.isnull().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)

def get_necessary_columns(data, col_list):
    return data[col_list]

def shuffle_missing(data, shuffle_column):
    data_shuffled = data.assign(shuffled = np.random.permutation(data[shuffle_column]))
    return data_shuffled

def calculate_categorical_proportions(data, categorical_column, target_column):
    groupby_filtered = data.groupby(categorical_column)[target_column].size().fillna(0)
    groupby_filtered = groupby_filtered / groupby_filtered.sum()
    return groupby_filtered

def calculate_tvd(proportions1, proportions2):
    return (proportions1 - proportions2).abs().sum() / 2

def calculate_tvd_between_nulls(data, categorical_column, target_column):
    groupby_not_null = data.groupby(categorical_column)[target_column].size().fillna(0)
    groupby_not_null = groupby_not_null / groupby_not_null.sum()
    data_null = data[data[target_column].isnull()]
    groupby_null = data_null.groupby(categorical_column)[target_column].size().fillna(0)
    groupby_null = groupby_null / groupby_null.sum()
    tvd = calculate_tvd(groupby_not_null, groupby_null)
    return tvd

In [284]:
def perform_missingness_permutation_test(data, categorical_column, target_column, n_repetitions=1000):
    filtered_data = get_necessary_columns(data, [categorical_column, target_column])
    observed_value = calculate_tvd_between_nulls(filtered_data, categorical_column, target_column)
    tvds = []

    for _ in range(n_repetitions):
        shuffled_data = shuffle_missing(filtered_data, target_column)
        tvd = calculate_tvd_between_nulls(shuffled_data, categorical_column, 'shuffled')
        tvds.append(tvd)
    
    return observed_value, tvds

def calculate_p_value(observed_statistic, permuted_vals):
    p_value = np.mean(np.array(permuted_vals) >= observed_statistic)
    return p_value

def plot_permutation_test_distribution(permuted_vals, observed_statistic, categorical_column_name, target_column_name, p_value=None):
    title = (
        f'Empirical Distribution of the TVD of {categorical_column_name} Proportion Differences<br>'
        f'Between Null and Not Null Values of {target_column_name}'
    )
    fig = px.histogram(pd.DataFrame(permuted_vals), x=0, nbins=50, histnorm='probability', 
                        title=title)
    fig.add_vline(x=observed_statistic, line_color='red', line_width=1, opacity=1)
    fig.add_annotation(text=f'<span style="color:red">Observed TVD = {round(observed_statistic, 2)}</span><br><span style="color:red">P-value = {round(p_value, 3)}</span>',
                   x=1.2 * observed_statistic, showarrow=False, y=0.05)
    return fig

In [285]:
observed_statistic, permuted_vals = perform_missingness_permutation_test(data, 'CAUSE.CATEGORY', 'OUTAGE.DURATION', n_repetitions=1000)
p_value = calculate_p_value(observed_statistic, permuted_vals)
print(f'P-value: {p_value}')
fig = plot_permutation_test_distribution(permuted_vals, observed_statistic, 'CAUSE.CATEGORY', 'OUTAGE.DURATION', p_value=p_value)
fig.show()

P-value: 0.0


In [286]:
observed_statistic, permuted_vals = perform_missingness_permutation_test(data, 'MONTH', 'OUTAGE.DURATION', n_repetitions=1000)
p_value = calculate_p_value(observed_statistic, permuted_vals)
print(f'P-value: {p_value}')
fig = plot_permutation_test_distribution(permuted_vals, observed_statistic, 'MONTH', 'OUTAGE.DURATION', p_value=p_value)
fig.show()

P-value: 0.086


## Step 4: Hypothesis Testing

In [287]:
# Define a mapping from detailed climate regions to 'North' and 'Not North'
climate_region_map = {'North': ['Northeast', 'East North Central', 'Northwest', 'West North Central'], 'Not North': ['South', 'West', 'Central', 'Southeast', 'Southwest']}
# Reverse the map to easily map each region to its bin
bin_map = {region: group for group, regions in climate_region_map.items() for region in regions}
# Map the CLIMATE.REGION to the binned version
data['CLIMATE.REGION.BINNED'] = data['CLIMATE.REGION'].map(bin_map)
# Determine the counts in each bin for sampling later on
binned_vals = data['CLIMATE.REGION.BINNED'].value_counts()
# Determine the observed mean for the 'North' bin
observed_mean = data.loc[data['CLIMATE.REGION.BINNED'] == 'North', 'OUTAGE.DURATION'].mean()
# Perform permutation test
n_repetitions = 1000
permuted_means = []
for _ in range(n_repetitions):
    sample_mean = data['OUTAGE.DURATION'].sample(n=binned_vals['North']).mean()
    permuted_means.append(sample_mean)
# Calculate p-value
p_value = calculate_p_value(observed_mean, permuted_means)
print(f'P-value: {p_value}')
# Plot the hypothesis test distribution
title = (
    f'Empirical Distribution of the Mean OUTAGE.DURATION<br>'
    f'For Random Samples of Size {binned_vals["North"]} from the Full Dataset'
)
fig = px.histogram(pd.DataFrame(permuted_means), x=0, nbins=50, histnorm='probability', 
                    title=title)
fig.add_vline(x=observed_mean, line_color='red', line_width=1, opacity=1)
fig.add_annotation(text=f'<span style="color:red">Observed Mean for NORTH = {round(observed_mean, 2)}</span><br><span style="color:red">P-value = {round(p_value, 3)}</span>',
                   x=1.2 * observed_mean, showarrow=False, y=0.05)

P-value: 0.002


## Step 5: Framing a Prediction Problem

In [288]:
# TODO

## Step 6: Baseline Model

In [289]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer

In [291]:
model_columns = ['YEAR', 'MONTH', 'POSTAL.CODE', 'NERC.REGION',
                    'CLIMATE.REGION', 'ANOMALY.LEVEL', 'CLIMATE.CATEGORY',
                    'HURRICANE.NAMES', 'PCT_LAND', 'PCT_WATER_TOT', 'PCT_WATER_INLAND']
model_data = data[model_columns]
categorical_cols = model_data.select_dtypes(include=['object']).columns.tolist()
numerical_cols = model_data.select_dtypes(include=['number']).columns.tolist()

categorical_cols

['POSTAL.CODE',
 'NERC.REGION',
 'CLIMATE.REGION',
 'CLIMATE.CATEGORY',
 'HURRICANE.NAMES']

In [292]:
missing_counts_model = model_data.isnull().sum()
missing_counts_model = missing_counts_model[missing_counts_model > 0].sort_values(ascending=False)

def prob_impute_categorical(series):
    series = series.copy()
    num_null = series.isnull().sum()
    fill_values = np.random.choice(series.dropna(), size=num_null, replace=True)
    series[series.isnull()] = fill_values
    return series

def prob_impute_numerical(series):
    return series.fillna(series.mean())

for col in model_data.columns:
    if model_data[col].dtype == 'object':
        model_data[col] = prob_impute_categorical(model_data[col])
    else:
        model_data[col] = prob_impute_numerical(model_data[col])

C:\Users\sweek\AppData\Local\Temp\ipykernel_48336\970413556.py:18: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\sweek\AppData\Local\Temp\ipykernel_48336\970413556.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\sweek\AppData\Local\Temp\ipykernel_48336\970413556.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/panda

In [293]:
col_trans = make_column_transformer(
    (OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols),
    (StandardScaler(), numerical_cols),
    remainder='passthrough',
)

pl = make_pipeline(col_trans, LinearRegression())

In [294]:
from sklearn.model_selection import train_test_split

X = model_data
y = prob_impute_numerical(data['OUTAGE.DURATION'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [295]:
pl.fit(X_train, y_train)
y_pred = pl.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f'RMSE: {rmse}')
print(f'R2: {r2}')

def create_residual_plot(y_true, y_pred):
    residuals = y_true - y_pred
    fig = px.scatter(x=y_pred, y=residuals, labels={'x': 'Predicted Values', 'y': 'Residuals'},
                    title='Residual Plot')
    fig.add_hline(y=0, line_color='red', line_width=1, opacity=1)
    return fig

residual_fig = create_residual_plot(y_test, y_pred)
residual_fig.show()

RMSE: 5473.032183489872
R2: -0.14369236233690308


c:\Users\sweek\miniforge3\envs\dsc80\Lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning:

Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros



## Step 7: Final Model

In [ ]:
# TODO

## Step 8: Fairness Analysis

In [ ]:
# TODO